In [18]:
import os 
import getpass
from dotenv import load_dotenv
import pydantic
from crewai import LLM, Agent, Task, Crew
from crewai_tools import PDFSearchTool
from chromadb.config import Settings
from crewai.rag.chromadb.config import ChromaDBConfig

llm = LLM(model="openai/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434/v1")

llm.call('Hello')



#
load_dotenv()

#
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

In [1]:
from crewai.rag.config.utils import set_rag_config, get_rag_client, clear_rag_config
from chromadb.utils.embedding_functions.openai_embedding_function import OpenAIEmbeddingFunction
from chromadb.utils.embedding_functions.ollama_embedding_function import OllamaEmbeddingFunction
from typing import Literal, cast
from crewai.rag.chromadb.types import ChromaEmbeddingFunctionWrapper
from dataclasses import field

# _chromaEmbeddingFunctionWrapper = cast(ChromaEmbeddingFunctionWrapper, OllamaEmbeddingFunction(model_name="all-minilm", url="http://localhost:11434"))
# r = OllamaEmbeddingFunction(model_name="all-minilm", url="http://localhost:11434").embed_query("Hello")

# def _default_embedding_function():
#     return cast(ChromaEmbeddingFunctionWrapper, OllamaEmbeddingFunction(model_name="all-minilm", url="http://localhost:11434"))

# _embedding_function = ChromaEmbeddingFunctionWrapper()

# ChromaDB (default)
from crewai.rag.chromadb.config import ChromaDBConfig

croma_client = ChromaDBConfig(dict(
    embedding_function=OpenAIEmbeddingFunction(model_name="all-minilm", api_base="http://localhost:11434/v1"),
    settings=dict()
))

set_rag_config(croma_client)
chromadb_client = get_rag_client()

# Example operations (same API for any provider)
client = chromadb_client
_collection_name="docs"
collection = client.get_or_create_collection(collection_name=_collection_name)
print(f"Vector store initialized. Collection: {_collection_name}")
print(f"Existing documents in collection: {collection.count()}")
client.add_documents(
    collection_name=_collection_name,
    documents=[{"id": "1", "content": "CrewAI enables collaborative AI agents."}],
)
results = client.search(collection_name=_collection_name, query="collaborative agents", limit=3)

clear_rag_config()  # optional reset

Vector store initialized. Collection: docs
Existing documents in collection: 0


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}} in upsert.

In [ ]:


pydantic.SkipValidation

# 1. Initialize the tool
pdf_search_tool = PDFSearchTool(
    pdf='/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf',
    config={
        "embedding_model": {
            # Supported providers: "openai", "azure", "google-generativeai", "google-vertex",
            # "voyageai", "cohere", "huggingface", "jina", "sentence-transformer",
            # "text2vec", "ollama", "openclip", "instructor", "onnx", "roboflow", "watsonx", "custom"
            "provider": "ollama",  # or: "google-generativeai", "cohere", "ollama", ...
            "config": {
                # Model identifier for the chosen provider. "model" will be auto-mapped to "model_name" internally.
                "model": "nomic-embed-text"
                # Optional: API key. If omitted, the tool will use provider-specific env vars
                # (e.g., OPENAI_API_KEY or EMBEDDINGS_OPENAI_API_KEY for OpenAI).
                # "api_key": "sk-...",

                # Provider-specific examples:
                # --- Google Generative AI ---
                # (Set provider="google-generativeai" above)
                # "model_name": "gemini-embedding-001",
                # "task_type": "RETRIEVAL_DOCUMENT",
                # "title": "Embeddings",

                # --- Cohere ---
                # (Set provider="cohere" above)
                # "model": "embed-english-v3.0",

                # --- Ollama (local) ---
                # (Set provider="ollama" above)
                # "model": "nomic-embed-text",
            }
        },
        "vectordb": {
                    "provider": "chromadb",  # or "qdrant"
                    "config": {
                        # For ChromaDB: pass "settings" (chromadb.config.Settings) or rely on defaults.
                        # Example (uncomment and import):
                        # from chromadb.config import Settings
                        "settings": Settings(
                           persist_directory="/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/vactor_store/chroma",
                        #   allow_reset=True
                        )

                        # For Qdrant: pass "vectors_config" (qdrant_client.models.VectorParams).
                        # Example (uncomment and import):
                        # from qdrant_client.models import VectorParams, Distance
                        # "vectors_config": VectorParams(size=384, distance=Distance.COSINE),

                        # Note: collection name is controlled by the tool (default: "rag_tool_collection"), not set here.
                    }
        }
    }
)

# 2. Define the Agent
researcher = Agent(
    role='Research Analyst',
    goal='Provide accurate information based on the PDF content',
    backstory='An expert in analyzing documents and extracting key skills.',
    tools=[pdf_search_tool],
    verbose=True,
    llm=llm
)

# 3. Define the Task
task = Task(
    description='Search the policy PDF to find the key skills.',
    expected_output='A summary of the key skills.',
    agent=researcher
)

# 4. Run the Crew
crew = Crew(agents=[researcher], tasks=[task])
result = crew.kickoff()


TypeError: BaseModel.validate() takes 2 positional arguments but 3 were given

In [9]:
from crewai_tools import RagTool
from crewai_tools.tools.rag import RagToolConfig, VectorDbConfig, ProviderSpec
from crewai.rag.embeddings.providers.ollama.types import OllamaProviderSpec

# Create a RAG tool with custom configuration

vectordb: VectorDbConfig = {
    "provider": "chromadb",
    "config": {
        #"persist_directory":"/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/vactor_store/chroma"
    }
}

embedding_model: OllamaProviderSpec = {
    "provider": "ollama",
    "config": {
        "model_name": "nomic-embed-text",
        "url": "http://localhost:11434/api/embeddings"
    }
}

config: RagToolConfig = {
    "vectordb": vectordb,
    "embedding_model": embedding_model
}

rag_tool = RagTool(config=config, summarize=True)



ValidationError: 1 validation error for RagTool
  Value error, An embedding function already exists in the collection configuration, and a new one is provided. If this is intentional, please embed documents separately. Embedding function conflict: new: ollama vs persisted: openai [type=value_error, input_value={'collection_name': 'rag_...logservice_port=50052))}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error

In [ ]:
rag_tool.add("Hello")